In [1]:
import argparse
import sys
from predict import run_predict

H:\anaconda3\envs\pytorch_cuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# for a give sequence and position, add "@" after if the position is "S" or "T"
def add_at_after_st(seq, position, unique_position):
    try:
        if seq[position-1] in ['S', 'T']:
            return seq[:position] + "@" + seq[position:]
    except:
        filename = "error_unique_position.txt"
        with open(filename, "a") as fasta_file:
            fasta_file.write(f"{unique_position}\n")            
    else:
        filename = "error_unique_position.txt"
        with open(filename, "a") as fasta_file:
            fasta_file.write(f"{unique_position}\n")  


In [3]:
# donwload sequence object based on uniprot id from entrez and return the sequence as a str
def download_uniprot_seq(uniprot_id):
    Entrez.email = "luvul@med.umich.edu"  # Replace with your email
    handle = Entrez.efetch(db="protein", id=uniprot_id, rettype="fasta", retmode="text")
    sequence_record = SeqIO.read(handle, "fasta")
    # protein_1433_dic[uniprot_id] = sequence_record
    handle.close()
    sequence = str(sequence_record.seq)
    return sequence

In [4]:
model_loc = "H:\PhosphoLingo_ST_new.ckpt"
dataset_fasta = "test.fasta"
output_file = "test.csv"

In [ ]:
run_predict(model_loc, dataset_fasta, output_file)

In [32]:
import pandas as pd

In [33]:
augmented_original_Lingo_score_df = pd.read_csv("augmented_original_Lingo_score_df.csv")

In [34]:
augmented_original_Lingo_score_df

,Unnamed: 0,level_0,Hit site,Hit accession,Hit sequence,label,augmented data?,index,position,unique_position,phosphoLingo_score
0,0,0,S467p,P43681,LAKARSLsVQHMSSP,1,0,NaN,467,P43681_467,0.874
1,1,1,S779p,P11362,PLDQYSPsFPDTRSS,1,0,NaN,779,P11362_779,0.269
2,2,2,T32p,P98177,QSRPRSCtWPLPRPE,1,0,NaN,32,P98177_32,0.827
3,3,3,S8p,P14136,MERRRITsAARRSYV,1,0,NaN,8,P14136_8,0.784
4,4,4,S640p,P08151,AGVTRRAsDPAQAAD,1,0,NaN,640,P08151_640,0.961
...,...,...,...,...,...,...,...,...,...,...,...
1549,1549,551,T100p,M3XT83,ARSPPRPtLAREDDD,0,1,984.0,100,M3XT83_100,NaN
1550,1550,552,S860p,M3XMD3,EEGAASGsNGNFPEG,0,1,985.0,860,M3XMD3_860,NaN
1551,1551,553,S14p,M3YH13,NTINRSSsFGNFDRF,1,1,986.0,14,M3YH13_14,NaN
1552,1552,554,S12p,M3YUM4,PSDHLLDsLEELGDN,0,1,987.0,12,M3YUM4_12,NaN


In [35]:
# get all the row which phosphoLingo_score is Nan
filtered_df = augmented_original_Lingo_score_df[augmented_original_Lingo_score_df['phosphoLingo_score'].isna()]

In [41]:
filtered_df.to_csv("no_phospholingo_score.csv")

In [42]:
filtered_df

,Unnamed: 0,level_0,Hit site,Hit accession,Hit sequence,label,augmented data?,index,position,unique_position,phosphoLingo_score
33,33,33,S68p,Q8IYK8,GAPRRRGsMPVPYKH,1,0,NaN,68,Q8IYK8_68,NaN
34,34,34,S333p,Q8IYK8,FFKQRSRsCHDLSVL,1,0,NaN,333,Q8IYK8_333,NaN
54,54,54,S230p,P17035,KKLTRRAsFSAQSAS,1,0,NaN,230,P17035_230,NaN
66,66,66,S51p,Q48BD8,PVLERSKsAPALLTA,1,0,NaN,51,Q48BD8_51,NaN
67,67,67,S51p,Q48BD8,PVLERAKsAPALLTA,1,0,NaN,51,Q48BD8_51,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1549,1549,551,T100p,M3XT83,ARSPPRPtLAREDDD,0,1,984.0,100,M3XT83_100,NaN
1550,1550,552,S860p,M3XMD3,EEGAASGsNGNFPEG,0,1,985.0,860,M3XMD3_860,NaN
1551,1551,553,S14p,M3YH13,NTINRSSsFGNFDRF,1,1,986.0,14,M3YH13_14,NaN
1552,1552,554,S12p,M3YUM4,PSDHLLDsLEELGDN,0,1,987.0,12,M3YUM4_12,NaN


In [39]:
uniprot_dict = {}

In [ ]:
from Bio import SeqIO
from Bio import Entrez
filename = "no_phospholingo_score.fasta"
for index, row in filtered_df.iterrows():
    uniprot_id = row["Hit accession"].split("-")[0]
    position = row["position"]
    unique_position = row["unique_position"]
    if uniprot_id in uniprot_dict:
        seq = uniprot_dict[uniprot_id]
    else:
        try:
            seq = download_uniprot_seq(uniprot_id)
            uniprot_dict[uniprot_id] = seq
        except:
            continue
    print(unique_position, len(seq))
    mut_seq = add_at_after_st(seq, position, unique_position)
    with open(filename, "a") as fasta_file:
        fasta_file.write(f">{unique_position}\n")
        fasta_file.write(f"{mut_seq}\n")

Q8IYK8_68 340
Q8IYK8_333 340
P17035_230 718
Q48BD8_51 447
Q48BD8_51 447
Q9V491_1794 1945
Q45VV3_168 395
Q29495_31 207
Q29495_205 207
Q9R1V6_855 904
Q9R1V6_832 904
P56402_256 271
P56402_269 271
Q60875_885 985
O55043_516 646
O55043_526 646
P06685_23 1023
Q61337_136 204
Q9QUN3_152 457
Q04982_769 766
Q04982_365 766
O88778_2845 3938
Q02294_2126 2336
P97756_74 505
Q8BGU5_100 341
Q923J1_1403 1863
O08785_845 855
Q07174_400 467
Q07174_443 467
P26955_603 896
Q9Z2F5_147 430
Q8VDF3_369 370
Q8VDF3_369 370
Q8VDF3_369 370
Q8VDF3_369 370
Q6S7F2_411 904
Q58FA4_395 860
Q9WVS8_486 806
Q6P4D5_29 195
Q8BGI4_322 693
Q86V87_325 743
Q9WVH4_32 672
Q9WVH3_197 505
Q9WVH3_32 505
O89100_254 322
P03995_8 430
P08050_373 382
P42260_846 908
P42260_868 908
Q60760_455 621
P97879_956 1112
P18266_9 420
P68431_10 136
P68431_10 136
P68431_28 136
O88704_867 910
O88704_810 910
Q8C2B3_178 938
Q8C2B3_479 938
Q8C2B3_344 938
P35569_637 1233
P81122_573 1321
Q9QWL7_44 433
Q9QWL7_9 433
Q6VV64_264 394
O88448_575 599
Q61097_297 873
Q6

In [ ]:
with open("uniprot.txt", "a") as fasta_file:
    fasta_file.write(uniprot_dict)